In [1]:
import numpy as np
import pandas as pd
import wandb
import random
import torch
import torch.nn as nn
import torch.optim as optim
import math
import datetime

from torch.optim import SGD, Optimizer
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter

from transformers import RobertaTokenizer, RobertaForSequenceClassification


In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/hutchii/.netrc.
wandb: Currently logged in as: hutchii (hutchtech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
def print_shapes(name, x, y):
      print(f"{name}\n{'-'*30}")
      print(f"x: {tuple(x.shape)}, y: {tuple(y.shape)}\n")

In [4]:
run = wandb.init(
    entity="hutchtech",
    project="MentalHealthClassifier",
    config = {
        "learning_rate":0.02,
        "dataset":"r/mentalhealth",
        "epochs":5
    },
)

config = {
    "epochs":5,
    "batch_size":16,
    "hf_access_token": "",
    "learning_rate":0.02
}

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /Users/hutchii/projects/MentalHealthLLM/notebooks/wandb/run-20260718_170133-wrm87rhi
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pious-frog-6
wandb: ⭐️ View project at https://wandb.ai/hutchtech/MentalHealthClassifier
wandb: 🚀 View run at https://wandb.ai/hutchtech/MentalHealthClassifier/runs/wrm87rhi


In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")
run_name = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(f"tb_logs/{run_name}")

Using device: mps


In [6]:
data = pd.read_csv("/Users/hutchii/projects/MentalHealthLLM/data/mh_sentiment_data.csv")
cleaned = data.dropna(subset=["statement"])
cleaned = cleaned.sample(frac=1, random_state=42).reset_index(drop=True)

statements = cleaned["statement"]
status = cleaned["status"]
split_idx = round(len(statements) * .75)

tokenizer = RobertaTokenizer.from_pretrained("FacebookAI/roberta-base", token=config["hf_access_token"])

In [7]:
print(statements[0])
print(status[0])

I'm lazy to complain about it ba ihh
Normal


In [8]:
x_tokens = tokenizer(list(statements), padding=False, truncation=True)

In [9]:
labels = (cleaned["status"].unique().tolist())
num_labels = len(labels)
print(num_labels)

stoi = {c:i for i,c in enumerate(labels)}
itos = {i:c for i,c in enumerate(labels)}
targets_encoded = torch.tensor(np.array([stoi[t] for t in status.tolist()]))

7


In [10]:
input_ids = x_tokens["input_ids"]

input_ids_train = input_ids[:split_idx]
y_train = targets_encoded[:split_idx]

input_ids_test = input_ids[split_idx:]
y_test = targets_encoded[split_idx:]

model = RobertaForSequenceClassification.from_pretrained("FacebookAI/roberta-base", num_labels=num_labels)
for p in model.roberta.parameters():
    p.requires_grad = False

model = model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, labels):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "labels": self.labels[idx]}

def collate_fn(batch):
    features = [{"input_ids": item["input_ids"]} for item in batch]
    padded = tokenizer.pad(features, return_tensors="pt")
    labels = torch.stack([item["labels"] for item in batch])
    return padded["input_ids"], padded["attention_mask"], labels

train_dataset = TextDataset(input_ids_train, y_train)
test_dataset = TextDataset(input_ids_test, y_test)

train_loader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=config["batch_size"], collate_fn=collate_fn)

test_loader = DataLoader(dataset=test_dataset, shuffle=True, batch_size=config["batch_size"], collate_fn=collate_fn)

In [14]:
optimizer = optim.AdamW(model.parameters(), lr=config["learning_rate"])

In [15]:
config["epochs"]

5

In [16]:
#model.train()
for e in tqdm(range(config["epochs"]), desc="Epochs"):
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {e+1} Batches")):
        b_input, b_attn, b_targets = batch
        b_input = b_input.to(device)
        b_attn = b_attn.to(device)
        b_targets = b_targets.to(device)

        print("forward pass")
        outputs = model(
            input_ids=b_input,
            attention_mask=b_attn,
            labels=b_targets
        )

        pred = outputs.logits.argmax(dim=-1)
        targets = (pred == b_targets).sum().item()

        print("calc grad")
        optimizer.zero_grad()
        loss = outputs.loss
        loss.backward()

        print("Updating weights")
        optimizer.step()

        print("write to wandb")
        print("\n")
        run.log({ "b_acc": targets / config["batch_size"], "b_loss":outputs.loss.item() })

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1 Batches:   0%|          | 0/2470 [00:00<?, ?it/s]

forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating

Epoch 2 Batches:   0%|          | 0/2470 [00:00<?, ?it/s]

forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating weights
write to wandb


forward pass
calc grad
Updating

KeyboardInterrupt: 